[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/farhad-abtahi/healthcareaibook/blob/main/vol%201%20notebooks/chapter_07/notebook_7_3_sepsis_prediction_xgboost.ipynb)

*Click the badge above to open this notebook in Google Colab (no setup required)*

---


# Notebook 7.3: Sepsis Prediction from Time Series Data

**Chapter 7: Time Series Analysis - Implementing Journey 1 (Marcus)**

**Journey Connection**: This notebook implements Journey 1 from Chapter 3, where an AI early-warning system correctly flagged Marcus's sepsis, but the alert was lost to alert fatigue. For the clinical context and patient story, refer to Chapter 3.

## Learning Objectives

By the end of this notebook, you will be able to:
1. Understand temporal feature engineering for sepsis early warning systems
2. Implement gradient boosting (XGBoost) for imbalanced classification
3. Optimize decision thresholds for sensitivity/specificity trade-offs
4. Visualize prediction timelines and analyze alert patterns
5. Understand why high AUROC doesn't guarantee clinical utility

## Clinical Context

Sepsis is a life-threatening condition where mortality increases ~8% per hour of delayed treatment. Early signs are subtle: slight increase in heart rate, mild fever, barely perceptible drop in blood pressure. Can AI detect these patterns before human clinicians?

**Marcus's story** (Chapter 3, Journey 1): Marcus, 68, a retired teacher with type 2 diabetes and mild chronic kidney disease, is recovering on the orthopedic ward after elective **hip** replacement. On day 2 at 2:47 AM the AI flags *"High sepsis risk: 73%"*, a correct early warning. But the night nurse has already received 12 similar alerts that shift (8 of them false alarms), so she plans to check on him after her medication pass. About 45 minutes later his blood pressure has dropped to 92/58 and he is confused; a rapid response follows. Marcus survives but needs 3 days in the ICU and an extra week in hospital. The AI technically succeeded (true positive) but operationally failed (delayed response due to alert fatigue).

**The challenge**: Sepsis prevalence is only about 2% of hospital admissions. At that prevalence, the Chapter 3 model's 82% sensitivity and 75% specificity produce roughly 15 false alarms for every true alert (we compute this in Section 7).

---

## Setup

In [1]:
# ── Setup (JupyterLite) ─────────────────────────────────────────────────────
import sys
if sys.platform == "emscripten":          # JupyterLite / Pyodide
    import micropip
    await micropip.install(["seaborn", "xgboost"])

In [2]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, classification_report
from sklearn.metrics import precision_recall_curve, average_precision_score
import xgboost as xgb
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

# Plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print("Libraries imported successfully!")
print(f"XGBoost version: {xgb.__version__}")

Libraries imported successfully!
XGBoost version: 3.2.0


## 1. Generate Synthetic ICU Time Series Data

We'll create synthetic vital signs data inspired by MIMIC-III patterns. In a real scenario, you would load actual EHR data.

**Data structure**:
- 4000 patient admissions (enough that the 15% test split still contains about a dozen septic patients at ~2% prevalence)
- ~2% develop sepsis (the real-world prevalence quoted in Chapter 3)
- Vital signs sampled every 15 minutes
- 48-hour observation window

**What makes it realistically hard** (Chapter 3: early signs "could indicate sepsis, or they could indicate pain, dehydration, anxiety, or nothing at all"):
- Every patient has their own baseline (one patient's normal heart rate is 65, another's 95) plus slow physiological drift
- Early sepsis is often subtle: deterioration strength varies between patients and accelerates only near onset
- About one in five non-septic patients has a transient **sepsis mimic** episode (pain, dehydration, post-operative fever) with similar vital-sign changes

These knobs (`severity_range`, `mimic_rate`, `mimic_severity`, `ramp_power`) were set so that, averaged over several random seeds, the **admission-level** alert performance measured in Section 7b lands near the real system described in Chapter 3 (82% sensitivity at 75% specificity, about 15 false alarms per true alert). A single run has only about a dozen septic test patients, so its numbers scatter around those values. Change the knobs to see how task difficulty drives alert burden.

**Censoring at onset**: the early-warning task ends when sepsis is recognised. Rows at or after onset are kept for the plots below but are **excluded** from training and evaluation (Section 4); they are not labelled negative.

In [ ]:
def _ar1(n, phi, sd):
    """Slow within-patient physiological drift (AR(1) process)."""
    e = np.random.normal(0, sd * np.sqrt(1 - phi ** 2), n)
    x = np.empty(n); x[0] = np.random.normal(0, sd)
    for i in range(1, n):
        x[i] = phi * x[i - 1] + e[i]
    return x


def generate_synthetic_icu_data(n_patients=4000, sepsis_rate=0.02, hours=48, sample_interval=0.25,
                                severity_range=(0.3, 1.0), mimic_rate=0.2, mimic_severity=(0.15, 0.7), ramp_power=1.5):
    """
    Generate SYNTHETIC ICU vital signs.

    Parameters:
    - n_patients: Number of patient admissions
    - sepsis_rate: Proportion who develop sepsis
    - hours: Observation window in hours
    - sample_interval: Sampling frequency in hours (0.25 = 15 min)
    - severity_range: how strongly a septic patient's vitals change (0 = not at all, 1 = full
      textbook picture); real early sepsis is often subtle
    - mimic_rate: fraction of NON-septic patients with a transient episode that looks like early
      sepsis (pain, dehydration, post-operative fever, anxiety: Chapter 3)
    - mimic_severity: strength of those mimic episodes
    - ramp_power: shape of the 12 h deterioration before onset (1 = linear; larger = changes
      concentrated in the last hours, so less warning time)

    Realism knobs (why the task is hard): patient-specific baselines, slow drift,
    subtle deterioration in some septic patients, and sepsis mimics in non-septic ones.

    Returns:
    - DataFrame with patient_id, time, vital signs, sepsis label and onset time
    """
    n_t = int(hours / sample_interval)
    t = np.arange(n_t) * sample_interval
    # vital: (population mean, between-patient SD, drift SD, measurement noise SD, change at full deterioration)
    spec = {'heart_rate': (80, 10, 5, 4, 25), 'bp_sys': (122, 14, 6, 6, -20), 'bp_dia': (74, 8, 4, 5, -10),
            'resp_rate': (16, 2.5, 1.5, 2, 8), 'temperature': (37.0, 0.3, 0.2, 0.15, 1.5),
            'spo2': (97, 1.2, 0.8, 1, -3)}
    limits = {'heart_rate': (40, 180), 'bp_sys': (70, 200), 'bp_dia': (40, 120), 'resp_rate': (8, 40),
              'temperature': (35, 41), 'spo2': (80, 100)}

    frames = []
    for patient_id in range(n_patients):
        develops_sepsis = np.random.rand() < sepsis_rate
        d = np.zeros(n_t)  # deterioration profile in [0, 1]
        onset = np.nan
        if develops_sepsis:
            # Sepsis onset somewhere between hour 12 and 40; vitals drift over the 12 h before
            onset = np.random.uniform(12, 40)
            severity = np.random.uniform(*severity_range)
            d = severity * np.clip(1 - (onset - t) / 12, 0, 1) ** ramp_power
        elif np.random.rand() < mimic_rate:
            # Transient sepsis mimic: rises and resolves over 4-12 h
            start, dur = np.random.uniform(0, hours - 4), np.random.uniform(4, 12)
            severity = np.random.uniform(*mimic_severity)
            d = severity * np.clip(1 - np.abs(t - (start + dur / 2)) / (dur / 2), 0, 1)

        rec = {'patient_id': patient_id, 'time_hours': t}
        for vital, (mu, between, drift, noise, delta) in spec.items():
            x = (np.random.normal(mu, between) + _ar1(n_t, 0.97, drift)
                 + delta * d + np.random.normal(0, noise, n_t))
            rec[vital] = np.clip(x, *limits[vital])

        # Label: sepsis within the next 6 hours (prediction target).
        # Rows at/after onset get 0 here but are EXCLUDED before training/evaluation (Section 4).
        rec['sepsis_within_6h'] = ((onset - t <= 6) & (t < onset)).astype(int) if develops_sepsis else np.zeros(n_t, int)
        rec['develops_sepsis'] = develops_sepsis
        rec['sepsis_onset_hour'] = onset
        frames.append(pd.DataFrame(rec))

    return pd.concat(frames, ignore_index=True)

# Generate data
print("Generating synthetic ICU data...")
df = generate_synthetic_icu_data(n_patients=4000, sepsis_rate=0.02)  # ~2% prevalence (Chapter 3)

print(f"\nDataset shape: {df.shape}")
print(f"Number of patients: {df['patient_id'].nunique()}")
print(f"Number of patients with sepsis: {df.groupby('patient_id')['develops_sepsis'].first().sum()}")
print(f"Percentage with sepsis: {df.groupby('patient_id')['develops_sepsis'].first().mean()*100:.1f}%")
print(f"\nClass distribution (6-hour prediction window):")
print(df['sepsis_within_6h'].value_counts())
print(f"Positive rate: {df['sepsis_within_6h'].mean()*100:.2f}%")

In [4]:
# Display sample data
print("Sample of raw data:")
df.head(20)

Sample of raw data:


,patient_id,time_hours,heart_rate,bp_sys,bp_dia,resp_rate,temperature,spo2,sepsis_within_6h,develops_sepsis
0,0,0.00,66.104959,123.189022,77.232330,19.031546,36.825737,97.474830,0,False
1,0,0.25,70.803607,110.759172,54.099608,18.851109,37.244934,96.476124,0,False
2,0,0.50,72.322822,112.575932,69.373250,9.581138,36.811158,98.597720,0,False
3,0,0.75,96.591436,123.942330,75.977753,14.453693,36.819924,98.947440,0,False
4,0,1.00,78.805873,113.644403,66.827582,15.514734,36.839905,97.994472,0,False
5,0,1.25,74.995759,123.893489,64.879047,19.275977,37.833494,99.193640,0,False
6,0,1.50,78.923934,128.817610,66.927317,11.250117,37.232110,97.461858,0,False
7,0,1.75,66.732640,111.194087,65.955582,16.403287,37.174637,98.887748,0,False
8,0,2.00,84.977871,127.549978,73.342673,14.129568,36.547554,99.099647,0,False
9,0,2.25,76.702630,115.896167,84.437731,13.305376,37.250439,98.296561,0,False


## 2. Exploratory Data Analysis

Let's visualize vital sign trajectories for septic vs non-septic patients.

In [5]:
# Select example patients for visualization
septic_patients = df[df['develops_sepsis'] == True]['patient_id'].unique()[:3]
normal_patients = df[df['develops_sepsis'] == False]['patient_id'].unique()[:3]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
vitals = ['heart_rate', 'bp_sys', 'temperature', 'resp_rate', 'spo2']
titles = ['Heart Rate (bpm)', 'Systolic BP (mmHg)', 'Temperature (°C)', 'Respiratory Rate (/min)', 'SpO2 (%)']

for idx, (vital, title) in enumerate(zip(vitals, titles)):
    ax = axes[idx // 3, idx % 3]

    # Plot septic patients
    for pid in septic_patients:
        patient_data = df[df['patient_id'] == pid]
        ax.plot(patient_data['time_hours'], patient_data[vital],
                color='red', alpha=0.6, linewidth=2)

    # Plot normal patients
    for pid in normal_patients:
        patient_data = df[df['patient_id'] == pid]
        ax.plot(patient_data['time_hours'], patient_data[vital],
                color='blue', alpha=0.6, linewidth=2)

    ax.set_xlabel('Time (hours)')
    ax.set_ylabel(title)
    ax.set_title(title)
    ax.legend(['Septic', 'Normal'], loc='best')
    ax.grid(True, alpha=0.3)

# Remove extra subplot
axes[1, 2].axis('off')

plt.tight_layout()
plt.suptitle('Vital Sign Trajectories: Septic (Red) vs Normal (Blue) Patients',
             y=1.02, fontsize=16, fontweight='bold')
plt.show()

print("\n📊 Observation: Notice how septic patients show gradual deterioration")
print("   - Heart rate trending upward")
print("   - Blood pressure trending downward")
print("   - Temperature increasing")
print("   - These trends are subtle and gradual!")


📊 Observation: Notice how septic patients show gradual deterioration
   - Heart rate trending upward
   - Blood pressure trending downward
   - Temperature increasing
   - These trends are subtle and gradual!


## 3. Feature Engineering

Raw vital signs alone are insufficient. We need to capture **temporal patterns** (Chapter 7, "Temporal Feature Engineering for Sepsis"):
- Current values
- Rolling statistics (mean, std, min, max) over windows of **2, 4 and 6 hours**
- Delta features (current value minus the value 2, 4 and 6 hours ago)
- Trend: the **slope over the past 4 hours** (least-squares fit, units per hour)
- Interaction features (shock index, MAP, pulse pressure)

With 15-minute sampling, a 2 h window is 8 samples, 4 h is 16 and 6 h is 24.

This is the **key to sepsis prediction**: detecting subtle deterioration patterns.

In [6]:
def create_temporal_features(df, windows_h=(2, 4, 6), slope_hours=4, sample_interval=0.25):
    """
    Create temporal features from time series vital signs.

    windows_h: rolling-window lengths in HOURS; converted to samples using
    sample_interval (0.25 h = 15 min, so 2 h = 8 samples, 4 h = 16, 6 h = 24).
    slope_hours: window for the least-squares trend (slope per hour).
    """

    df = df.sort_values(['patient_id', 'time_hours']).copy()

    vitals = ['heart_rate', 'bp_sys', 'bp_dia', 'resp_rate', 'temperature', 'spo2']

    features = df.copy()

    pid = df['patient_id']
    t = df['time_hours']

    def roll_mean(series, window, min_periods):
        return series.groupby(pid).rolling(window, min_periods=min_periods).mean().reset_index(0, drop=True)

    for vital in vitals:
        # Calculate features for each window
        for wh in windows_h:
            window = int(round(wh / sample_interval))  # hours -> samples
            window_hours = f'{wh:g}'

            # Rolling statistics
            features[f'{vital}_mean_{window_hours}h'] = (
                df.groupby('patient_id')[vital]
                .rolling(window, min_periods=1).mean()
                .reset_index(0, drop=True)
            )

            features[f'{vital}_std_{window_hours}h'] = (
                df.groupby('patient_id')[vital]
                .rolling(window, min_periods=1).std()
                .reset_index(0, drop=True)
            )

            features[f'{vital}_min_{window_hours}h'] = (
                df.groupby('patient_id')[vital]
                .rolling(window, min_periods=1).min()
                .reset_index(0, drop=True)
            )

            features[f'{vital}_max_{window_hours}h'] = (
                df.groupby('patient_id')[vital]
                .rolling(window, min_periods=1).max()
                .reset_index(0, drop=True)
            )

            # Delta (change from window ago)
            features[f'{vital}_delta_{window_hours}h'] = (
                df.groupby('patient_id')[vital].diff(window)
            )

        # Trend: least-squares slope over the past `slope_hours` (units per hour)
        # slope = cov(t, x) / var(t), computed with rolling means per patient
        w = int(round(slope_hours / sample_interval))
        x = df[vital]
        mp = w // 2  # need >= 2 h of data before reporting a 4 h slope
        mt, mx = roll_mean(t, w, mp), roll_mean(x, w, mp)
        cov_tx = roll_mean(t * x, w, mp) - mt * mx
        var_t = roll_mean(t * t, w, mp) - mt ** 2
        features[f'{vital}_slope_{slope_hours}h'] = cov_tx / var_t.replace(0, np.nan)

    # Interaction features
    features['shock_index'] = features['heart_rate'] / (features['bp_sys'] + 1e-6)
    features['map'] = (features['bp_sys'] + 2 * features['bp_dia']) / 3
    features['pulse_pressure'] = features['bp_sys'] - features['bp_dia']

    # Fill NaN from rolling/diff operations at the start of each stay with 0.
    # (Back-filling would copy FUTURE values into earlier rows: a subtle leak.)
    features = features.fillna(0)

    return features

print("Creating temporal features...")
df_features = create_temporal_features(df, windows_h=(2, 4, 6), slope_hours=4)  # 2h/4h/6h windows = 8/16/24 samples

print(f"\nOriginal features: {df.shape[1]}")
print(f"Engineered features: {df_features.shape[1]}")
print(f"Added features: {df_features.shape[1] - df.shape[1]}")

# Show some engineered features
feature_cols = ([c for c in df_features.columns if c.startswith('heart_rate_') and ('mean' in c or 'delta' in c)]
                + [c for c in df_features.columns if 'slope' in c][:2])
print(f"\nSample of engineered features:")
df_features[['patient_id', 'time_hours'] + feature_cols].head()

Creating temporal features...



Original features: 10
Engineered features: 103
Added features: 93

Sample of engineered features:


,patient_id,time_hours,heart_rate_mean_1.0h,heart_rate_delta_1.0h,heart_rate_mean_2.0h,heart_rate_delta_2.0h,heart_rate_mean_3.0h,heart_rate_delta_3.0h,bp_sys_mean_1.0h,bp_sys_delta_1.0h,bp_sys_mean_2.0h,bp_sys_delta_2.0h
0,0,0.00,66.104959,12.700914,66.104959,18.872912,66.104959,20.039935,123.189022,-9.544619,123.189022,4.360956
1,0,0.25,68.454283,12.700914,68.454283,18.872912,68.454283,20.039935,116.974097,-9.544619,116.974097,4.360956
2,0,0.50,69.743796,12.700914,69.743796,18.872912,69.743796,20.039935,115.508042,-9.544619,115.508042,4.360956
3,0,0.75,76.455706,12.700914,76.455706,18.872912,76.455706,20.039935,117.616614,-9.544619,117.616614,4.360956
4,0,1.00,79.630935,12.700914,76.925740,18.872912,76.925740,20.039935,115.230459,-9.544619,116.822172,4.360956


## 4. Prepare Training Data

**Important**: We split **by patient** (all rows of a patient go to one set) to avoid leakage between overlapping windows of the same patient, and **stratify by patient outcome** so that each set gets its share of the rare septic patients (at ~2% prevalence an unstratified split can leave only one or two septic patients in the test set). Each septic admission is also **censored at onset**: rows at or after onset are dropped (the patient is now being treated, and their vitals still look septic, so calling those rows "negative" would be wrong). This is a *patient-level* split, not a temporal one: a true temporal split would train on earlier admissions and test on later ones.

In [ ]:
# Censor each septic admission at onset: rows at/after onset are excluded, not labelled negative
df_features['develops_sepsis'] = df_features['develops_sepsis'].astype(bool)
post_onset = df_features['develops_sepsis'] & (df_features['time_hours'] >= df_features['sepsis_onset_hour'])
df_features = df_features[~post_onset].reset_index(drop=True)
print(f"Excluded {post_onset.sum()} post-onset rows from {df_features.loc[df_features['develops_sepsis'], 'patient_id'].nunique()} septic admissions")

# Separate features and labels
exclude_cols = ['patient_id', 'time_hours', 'sepsis_within_6h', 'develops_sepsis', 'sepsis_onset_hour']
feature_cols = [col for col in df_features.columns if col not in exclude_cols]

X = df_features[feature_cols]
y = df_features['sepsis_within_6h']
patient_ids = df_features['patient_id']

print(f"Feature matrix shape: {X.shape}")
print(f"Number of features: {len(feature_cols)}")
print(f"\nClass distribution:")
print(y.value_counts())
print(f"Positive rate: {y.mean()*100:.2f}%")

In [8]:
# Patient-level split, stratified by outcome: 70% train, 15% validation, 15% test
patient_outcome = df_features.groupby('patient_id')['develops_sepsis'].first()
train_patients, rest_patients = train_test_split(
    patient_outcome.index, test_size=0.30, stratify=patient_outcome.values, random_state=42)
val_patients, test_patients = train_test_split(
    rest_patients, test_size=0.50, stratify=patient_outcome.loc[rest_patients].values, random_state=42)

train_mask = patient_ids.isin(train_patients)
val_mask = patient_ids.isin(val_patients)
test_mask = patient_ids.isin(test_patients)

X_train, y_train = X[train_mask], y[train_mask]
X_val, y_val = X[val_mask], y[val_mask]
X_test, y_test = X[test_mask], y[test_mask]

print(f"Training set: {X_train.shape[0]} samples, {y_train.sum()} positive ({y_train.mean()*100:.2f}%)")
print(f"Validation set: {X_val.shape[0]} samples, {y_val.sum()} positive ({y_val.mean()*100:.2f}%)")
print(f"Test set: {X_test.shape[0]} samples, {y_test.sum()} positive ({y_test.mean()*100:.2f}%)")
for name, pats in [("train", train_patients), ("val", val_patients), ("test", test_patients)]:
    print(f"  septic patients in {name}: {int(patient_outcome.loc[pats].sum())} of {len(pats)}")

Training set: 134400 samples, 648 positive (0.48%)
Validation set: 28800 samples, 72 positive (0.25%)
Test set: 28800 samples, 168 positive (0.58%)


## 5. Train XGBoost Model

**Why XGBoost for sepsis prediction?**
- Handles tabular data excellently
- Naturally handles missing data
- Provides feature importance
- Captures complex nonlinear relationships
- Can use class weights to handle imbalance

In [9]:
# Calculate scale_pos_weight for class imbalance
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"Class imbalance ratio (negative:positive): {scale_pos_weight:.1f}:1")
print(f"Using scale_pos_weight={scale_pos_weight:.2f} to handle imbalance\n")

# Train XGBoost
print("Training XGBoost model...")
model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    eval_metric='auc'
)

model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)

print("✓ Model training complete!")

Class imbalance ratio (negative:positive): 206.4:1
Using scale_pos_weight=206.41 to handle imbalance

Training XGBoost model...


✓ Model training complete!


## 6. Model Evaluation

Let's evaluate using multiple metrics to understand the model's performance.

In [ ]:
# Generate predictions
y_train_pred_proba = model.predict_proba(X_train)[:, 1]
y_val_pred_proba = model.predict_proba(X_val)[:, 1]
y_test_pred_proba = model.predict_proba(X_test)[:, 1]

# Calculate AUROC
train_auc = roc_auc_score(y_train, y_train_pred_proba)
val_auc = roc_auc_score(y_val, y_val_pred_proba)
test_auc = roc_auc_score(y_test, y_test_pred_proba)

print("="*60)
print("MODEL PERFORMANCE - ROW-LEVEL AUROC (per 15-minute prediction)")
print("="*60)
print(f"Training AUROC:   {train_auc:.3f}")
print(f"Validation AUROC: {val_auc:.3f}")
print(f"Test AUROC:       {test_auc:.3f}")
print("="*60)

# Plot ROC curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC Curve
ax = axes[0]
for name, y_true, y_pred in [("Train", y_train, y_train_pred_proba),
                               ("Val", y_val, y_val_pred_proba),
                               ("Test", y_test, y_test_pred_proba)]:
    fpr, tpr, _ = roc_curve(y_true, y_pred)
    auc = roc_auc_score(y_true, y_pred)
    ax.plot(fpr, tpr, linewidth=2, label=f'{name} (AUC={auc:.3f})')

ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate (Sensitivity)')
ax.set_title('ROC Curve')
ax.legend()
ax.grid(True, alpha=0.3)

# Precision-Recall Curve
ax = axes[1]
for name, y_true, y_pred in [("Train", y_train, y_train_pred_proba),
                               ("Val", y_val, y_val_pred_proba),
                               ("Test", y_test, y_test_pred_proba)]:
    precision, recall, _ = precision_recall_curve(y_true, y_pred)
    ap = average_precision_score(y_true, y_pred)
    ax.plot(recall, precision, linewidth=2, label=f'{name} (AP={ap:.3f})')

baseline = y_test.mean()
ax.axhline(baseline, color='k', linestyle='--', linewidth=1, label=f'Baseline ({baseline:.3f})')
ax.set_xlabel('Recall (Sensitivity)')
ax.set_ylabel('Precision (PPV)')
ax.set_title('Precision-Recall Curve')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Threshold Optimization

**Critical question**: What probability threshold should trigger an alert?

- Too low (e.g., 0.1) → Many false positives → Alert fatigue
- Too high (e.g., 0.8) → Miss true cases → Patients die

We need to balance sensitivity (catching sepsis) with specificity (avoiding false alarms).

In [ ]:
def analyze_threshold(y_true, y_pred_proba, threshold):
    """Calculate metrics at a given threshold."""
    y_pred = (y_pred_proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    ppv = tp / (tp + fp) if (tp + fp) > 0 else 0
    npv = tn / (tn + fn) if (tn + fn) > 0 else 0

    fp_ratio = fp / tp if tp > 0 else float('inf')

    return {
        'threshold': threshold,
        'sensitivity': sensitivity,
        'specificity': specificity,
        'ppv': ppv,
        'npv': npv,
        'tp': tp,
        'fp': fp,
        'tn': tn,
        'fn': fn,
        'fp_ratio': fp_ratio
    }

# Analyze multiple thresholds
thresholds_to_test = [0.005, 0.01, 0.02, 0.05, 0.1, 0.2, 0.3, 0.5]
results = []

for thresh in thresholds_to_test:
    result = analyze_threshold(y_test, y_test_pred_proba, thresh)
    results.append(result)

results_df = pd.DataFrame(results)

print("\n" + "="*80)
print("THRESHOLD ANALYSIS (ROW LEVEL: each 15-minute prediction is one case)")
print("="*80)
print(results_df.to_string(index=False))
print("="*80)

print("\n💡 KEY INSIGHT: The False Positive Ratio (row level)")
print(f"   At threshold 0.5: {results_df[results_df['threshold']==0.5]['fp_ratio'].values[0]:.1f}:1 false-positive rows per true-positive row")
print(f"   At threshold 0.3: {results_df[results_df['threshold']==0.3]['fp_ratio'].values[0]:.1f}:1 false-positive rows per true-positive row")
fp_ratio_05 = results_df[results_df['threshold']==0.5]['fp_ratio'].values[0]
if fp_ratio_05 < 1:
    print(f"\n   Here there are FEWER false alarms than true alarms (test AUROC {test_auc:.3f}):")
    print("   the synthetic patients are far easier to separate than real ones.")
    print("   Real deployments have reported much higher FP:TP ratios (see the Epic example below),")
    print("   which is why high AUROC doesn't guarantee clinical utility.")
else:
    print("\n   This is why high AUROC doesn't guarantee clinical utility!")

# Chapter 3 arithmetic: Marcus's real system at real-world prevalence
prev, sens_book, spec_book, n = 0.02, 0.82, 0.75, 1000
tp_b = n * prev * sens_book
fp_b = n * (1 - prev) * (1 - spec_book)
print(f"\n📖 Chapter 3 system (sensitivity {sens_book:.0%}, specificity {spec_book:.0%}) at {prev:.0%} prevalence, per {n} patients:")
print(f"   {n*prev:.0f} have sepsis → {tp_b:.1f} caught; {n*(1-prev):.0f} do not → {fp_b:.0f} false alarms")
print(f"   ≈ {fp_b/tp_b:.0f} false alarms per true alert (PPV {tp_b/(tp_b+fp_b):.1%})")
print(f"   Chapter 3 counts ADMISSIONS ({prev:.0%} septic); the table above counts 15-minute ROWS ({y_test.mean():.2%} positive).")
print("   The two cannot be mixed: Section 7b evaluates our model per admission.")

In [12]:
# Visualize threshold trade-offs
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Sensitivity vs Specificity
ax = axes[0]
ax.plot(results_df['threshold'], results_df['sensitivity'], 'o-', linewidth=2, markersize=8, label='Sensitivity')
ax.plot(results_df['threshold'], results_df['specificity'], 's-', linewidth=2, markersize=8, label='Specificity')
ax.axhline(0.80, color='r', linestyle='--', alpha=0.5, label='Target: 80%')
ax.set_xlabel('Threshold')
ax.set_ylabel('Metric Value')
ax.set_title('Sensitivity vs Specificity Trade-off')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim([0, 1.05])

# False Positive Ratio
ax = axes[1]
fp_ratios = results_df['fp_ratio'].replace([np.inf], 100)  # Cap infinity at 100 for plotting
ax.plot(results_df['threshold'], fp_ratios, 'o-', linewidth=2, markersize=8, color='red')
ax.set_xlabel('Threshold')
ax.set_ylabel('False Positive Ratio (FP:TP)')
ax.set_title('False Alarms per True Positive')
ax.grid(True, alpha=0.3)
ax.axhline(10, color='orange', linestyle='--', alpha=0.5, label='10:1 ratio')
ax.legend()

plt.tight_layout()
plt.show()

print("\n⚠️  CLINICAL REALITY:")
print(f"   Even with a test AUROC of {test_auc:.2f}, you must choose:")
print("   - High sensitivity (catch more sepsis) → More false alarms → Alert fatigue")
print("   - High specificity (fewer false alarms) → Miss some sepsis cases → Harm")
print("   There is no perfect threshold!")


⚠️  CLINICAL REALITY:
   Even with a test AUROC of 1.00, you must choose:
   - High sensitivity (catch more sepsis) → More false alarms → Alert fatigue
   - High specificity (fewer false alarms) → Miss some sepsis cases → Harm
   There is no perfect threshold!


## 7b. The Chapter 3 Operating Point, per Admission

Chapter 3 reports Marcus's system at **82% sensitivity and 75% specificity** (AUROC 0.83) with sepsis in ~2% of **admissions**, which means **~15 false alarms per true alert** (PPV ≈ 6%), and a median lead time of **4.2 hours**. Those numbers count patients, so we must evaluate per patient too. Row-level sensitivity and specificity (Section 7) answer a different question: only a fraction of a percent of 15-minute rows are positive.

**Alert policy** (what the nurse actually experiences):
- An alert **fires** when the risk score stays at or above the threshold for 2 consecutive predictions (30 minutes)
- After an alert, that patient cannot alert again for 8 hours (refractory period)
- An alert is **true** if sepsis onset follows within 12 hours, otherwise **false**
- Each septic admission is censored at onset (Section 4)

**Threshold**: chosen on the **validation** admissions so that 75% of non-septic admissions never alert (75% admission-level specificity).

**Measured on the test admissions**:
- Sensitivity: septic admissions with a true alert before onset (missed patients are counted and reported)
- Specificity: non-septic admissions that never alert
- PPV and false alarms per true alert, both per admission (comparable with Chapter 3) and per individual alert
- Alerts per 100 patient-days
- **Lead time** = onset − first true alert, for detected patients only. The book's "time to clinical recognition" is approximated by the simulated onset time

The test set has only about a dozen septic patients, so sensitivity and lead time are noisy: the counts and an interval are printed so you can judge.

In [ ]:
# ── Alert policy: one decision per ADMISSION, not per 15-minute row ─────────
PERSIST = 2         # score must be above threshold for 2 consecutive predictions (30 min)
REFRACTORY_H = 8    # after an alert fires, that patient cannot alert again for 8 h
HORIZON_H = 12      # an alert is TRUE if sepsis onset follows within 12 h, otherwise FALSE


def fire_alerts(times, scores, thr):
    """Times at which the alert policy fires during one admission."""
    fired, run, last = [], 0, -np.inf
    for t, s in zip(times, scores):
        run = run + 1 if s >= thr else 0
        if run >= PERSIST and t - last >= REFRACTORY_H:
            fired.append(t)
            last = t
    return np.array(fired)


def sustained_peak(p):
    """Highest score held for PERSIST consecutive predictions: the admission alerts iff this >= threshold."""
    return p.rolling(PERSIST).min().max()


def evaluate_alerts(d, thr):
    """One row per admission: alerts fired, how many were true, and lead time of the first true alert."""
    out = []
    for pid, g in d.groupby('patient_id'):
        septic, onset = bool(g['develops_sepsis'].iloc[0]), g['sepsis_onset_hour'].iloc[0]
        fired = fire_alerts(g['time_hours'].values, g['p'].values, thr)
        true = fired[fired >= onset - HORIZON_H] if septic else fired[:0]
        peak = sustained_peak(g.loc[g['time_hours'] >= onset - HORIZON_H, 'p'] if septic else g['p'])
        out.append({'septic': septic, 'n_alerts': len(fired), 'n_true': len(true), 'peak': peak,
                    'lead_h': onset - true[0] if len(true) else np.nan, 'days': len(g) * 0.25 / 24})
    return pd.DataFrame(out)


id_cols = ['patient_id', 'time_hours', 'develops_sepsis', 'sepsis_onset_hour']
val_adm = df_features.loc[val_mask, id_cols].assign(p=y_val_pred_proba)
test_adm = df_features.loc[test_mask, id_cols].assign(p=y_test_pred_proba)

# Threshold chosen on VALIDATION admissions: 75% of non-septic admissions must never alert
val_peaks = val_adm[~val_adm['develops_sepsis']].groupby('patient_id')['p'].apply(sustained_peak)
alert_thr = np.quantile(val_peaks, 0.75)

# Everything below is measured on TEST admissions
adm = evaluate_alerts(test_adm, alert_thr)
sep, non = adm[adm['septic']], adm[~adm['septic']]
n_sep_test, n_non_test = len(sep), len(non)
n_detected = int((sep['n_true'] > 0).sum())
n_missed = n_sep_test - n_detected
n_fa_adm = int((non['n_alerts'] > 0).sum())
sens_adm, spec_adm = n_detected / n_sep_test, 1 - n_fa_adm / n_non_test
ppv_adm = n_detected / max(n_detected + n_fa_adm, 1)
fa_adm_per_tp = n_fa_adm / n_detected if n_detected else float('inf')
n_true_alerts = int(adm['n_true'].sum())
n_false_alerts = int(adm['n_alerts'].sum()) - n_true_alerts
ppv_alert = n_true_alerts / max(n_true_alerts + n_false_alerts, 1)
fa_per_true_alert = n_false_alerts / n_true_alerts if n_true_alerts else float('inf')
alerts_per_100pd = 100 * adm['n_alerts'].sum() / adm['days'].sum()
lead_median = sep['lead_h'].median()
auc_adm = roc_auc_score(adm['septic'], adm['peak'])
prev_adm = n_sep_test / len(adm)

print(f"Alert policy: score ≥ {alert_thr:.4f} for {PERSIST} consecutive predictions; {REFRACTORY_H} h refractory period;")
print(f"an alert counts as true if sepsis onset follows within {HORIZON_H} h. Threshold set on validation admissions.\n")
print(f"ADMISSION LEVEL ({len(adm)} synthetic test admissions: {n_sep_test} septic, {n_non_test} non-septic)")
rows = [('Sepsis prevalence (admissions)', '≈2%', f'{prev_adm:.1%}'),
        ('AUROC (admission level)', '0.83', f'{auc_adm:.2f}'),
        ('Specificity (non-septic admissions never alerted)', '75%', f'{spec_adm:.0%} ({n_non_test - n_fa_adm}/{n_non_test})'),
        ('Sensitivity (septic admissions alerted before onset)', '82%', f'{sens_adm:.0%} ({n_detected}/{n_sep_test}; {n_missed} missed)'),
        ('PPV (alerted admissions that are septic)', '≈6%', f'{ppv_adm:.1%}'),
        ('False-alarm admissions per detected sepsis', '≈15', f'{fa_adm_per_tp:.1f} ({n_fa_adm}/{n_detected})'),
        ('Median lead time (onset − first true alert)', '4.2 h',
         f'{lead_median:.1f} h ({n_detected} detected; {n_missed} missed not included)' if n_detected else 'n/a (none detected)'),
        ('PPV of an individual alert', '', f'{ppv_alert:.1%} ({n_true_alerts} true, {n_false_alerts} false alerts)'),
        ('False alerts per true alert', '', f'{fa_per_true_alert:.1f}'),
        ('Alerts per 100 patient-days', '', f'{alerts_per_100pd:.0f}')]
print(f"{'Quantity':<54} {'Chapter 3':>10}   {'This notebook (synthetic)'}")
print("-" * 105)
for q, book, ours in rows:
    print(f"{q:<54} {book:>10}   {ours}")
print("Lead time: 'clinical recognition' is approximated by the simulated onset time (used for evaluation only, never as a feature).")

# Row-level view at the SAME threshold, with the row-level prevalence (not comparable with Chapter 3)
op = analyze_threshold(y_test, y_test_pred_proba, alert_thr)
print(f"\nROW LEVEL (each 15-minute prediction is a case; positives = rows within 6 h before onset = {y_test.mean():.2%} of rows)")
print(f"   sensitivity {op['sensitivity']:.0%}, specificity {op['specificity']:.0%}, PPV {op['ppv']:.1%}, "
      f"{op['fp_ratio']:.0f} false-positive rows per true-positive row")

print("\n💡 Interpretation:")
if op['specificity'] > spec_adm + 0.05:
    print(f"   The same threshold has {op['specificity']:.0%} specificity per row but only {spec_adm:.0%} per admission: every admission")
    print(f"   has up to {int(48 / 0.25)} chances to cross it, so {1 - spec_adm:.0%} of non-septic patients alert at least once.")
z = 1.96  # 95% Wilson interval for sensitivity: with so few septic test patients it is wide
centre = (sens_adm + z**2 / (2 * n_sep_test)) / (1 + z**2 / n_sep_test)
half = z * np.sqrt(sens_adm * (1 - sens_adm) / n_sep_test + z**2 / (4 * n_sep_test**2)) / (1 + z**2 / n_sep_test)
print(f"   Sensitivity {sens_adm:.0%} rests on {n_sep_test} septic test patients (95% interval {centre - half:.0%}–{centre + half:.0%}): "
      + ("consistent with Chapter 3's 82%." if centre - half <= 0.82 <= centre + half else "NOT consistent with Chapter 3's 82% on this run."))
print(f"   Specificity {spec_adm:.0%} vs the 75% targeted on validation: "
      + ("the threshold transferred to the test set." if abs(spec_adm - 0.75) <= 0.04 else "the threshold did not transfer exactly to the test set."))
if n_detected:
    if 10 <= fa_adm_per_tp <= 20:
        print(f"   {fa_adm_per_tp:.0f} false-alarm admissions per detected sepsis: the same alert burden as Chapter 3 (≈15).")
    else:
        print(f"   {fa_adm_per_tp:.0f} false-alarm admissions per detected sepsis: {'heavier' if fa_adm_per_tp > 20 else 'lighter'} than Chapter 3's ≈15 on this run.")
    print(f"   Median lead time {lead_median:.1f} h vs 4.2 h in Chapter 3: "
          + ("similar." if abs(lead_median - 4.2) <= 1 else "different on this run (few patients, simulated onset).")
          + f" It describes only the {n_detected} detected patients.")
else:
    print("   No septic test patient was detected, so alert burden and lead time cannot be estimated on this run.")
if op['fp_ratio'] > 2 * fa_adm_per_tp:
    print(f"   Unit of analysis matters: per row the burden looks like {op['fp_ratio']:.0f}:1 because only {y_test.mean():.2%} of rows are positive.")
    print("   Row-level sensitivity/specificity must never be combined with the 2% prevalence of admissions.")

## 8. Feature Importance

Which features are most important for sepsis prediction?

In [13]:
# Get feature importance
importance_df = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

# Plot top 20 features
plt.figure(figsize=(12, 8))
top_features = importance_df.head(20)
plt.barh(range(len(top_features)), top_features['importance'])
plt.yticks(range(len(top_features)), top_features['feature'])
plt.xlabel('Feature Importance')
plt.title('Top 20 Most Important Features for Sepsis Prediction')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print("\nTop 10 Most Important Features:")
print(importance_df.head(10).to_string(index=False))

def feature_group(f):
    if '_delta_' in f: return 'delta (change)'
    if '_slope_' in f: return 'slope (trend)'
    if any(f'_{s}_' in f for s in ['mean', 'std', 'min', 'max']): return 'rolling window'
    if f in ['shock_index', 'map', 'pulse_pressure']: return 'interaction'
    return 'raw value'
group_share = importance_df.groupby(importance_df['feature'].map(feature_group))['importance'].sum()
vital_share = importance_df.groupby(importance_df['feature'].str.extract(
    r'^(heart_rate|bp_sys|bp_dia|resp_rate|temperature|spo2)')[0])['importance'].sum()

print("\n📊 Interpretation (importance share by feature type):")
for g, s in group_share.sort_values(ascending=False).items():
    print(f"   - {g:15s}: {s:.1%}")
print(f"   - Most important vital sign overall: {vital_share.idxmax()} ({vital_share.max():.0%} of importance)")
top_group = group_share.idxmax()
if top_group == 'delta (change)':
    print("   - Change (delta) features dominate, consistent with 'change matters more than absolute values'")
else:
    print(f"   - '{top_group}' features dominate ({group_share.max():.0%}); explicit change (delta)")
    print(f"     features carry only {group_share.get('delta (change)', 0):.1%} of the importance here")
print("   - Note: in synthetic data, the importances mirror how the generator encodes sepsis")


Top 10 Most Important Features:
              feature  importance
 heart_rate_mean_3.0h    0.555453
 heart_rate_mean_2.0h    0.370610
temperature_mean_3.0h    0.012857
 temperature_max_3.0h    0.010889
        spo2_min_2.0h    0.010597
  resp_rate_mean_3.0h    0.003464
  resp_rate_mean_2.0h    0.002487
  heart_rate_min_3.0h    0.001914
 temperature_std_3.0h    0.001133
  heart_rate_max_3.0h    0.001109

📊 Interpretation (importance share by feature type):
   - rolling window : 99.6%
   - delta (change) : 0.3%
   - raw value      : 0.1%
   - interaction    : 0.0%
   - Most important vital sign overall: heart_rate (93% of importance)
   - 'rolling window' features dominate (100%); explicit change (delta)
     features carry only 0.3% of the importance here
   - Note: in synthetic data, the importances mirror how the generator encodes sepsis


## 9. Patient-Level Prediction Timeline

Let's visualize how the model's predictions evolve over time for specific patients.

In [14]:
# Get predictions for all test patients
test_data = df_features[test_mask].copy()
test_data['pred_proba'] = y_test_pred_proba
test_data['prediction'] = (y_test_pred_proba >= alert_thr).astype(int)  # Section 7b operating point

# Find a true positive case (correctly identified sepsis)
tp_patients = test_data[(test_data['sepsis_within_6h'] == 1) &
                        (test_data['prediction'] == 1)]['patient_id'].unique()

# Find a false positive case
fp_patients = test_data[(test_data['sepsis_within_6h'] == 0) &
                        (test_data['prediction'] == 1)]['patient_id'].unique()

if len(tp_patients) > 0 and len(fp_patients) > 0:
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))

    # True Positive case
    tp_patient = tp_patients[0]
    tp_data = test_data[test_data['patient_id'] == tp_patient]

    ax = axes[0, 0]
    ax.plot(tp_data['time_hours'], tp_data['pred_proba'], 'b-', linewidth=2)
    ax.axhline(alert_thr, color='r', linestyle='--', label=f'Alert threshold ({alert_thr:.3f})')
    ax.fill_between(tp_data['time_hours'], 0, 1,
                     where=tp_data['sepsis_within_6h']==1,
                     alpha=0.2, color='red', label='True sepsis window')
    ax.set_xlabel('Time (hours)')
    ax.set_ylabel('Sepsis Probability')
    ax.set_title(f'True Positive: Patient {tp_patient}')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_yscale('log'); ax.set_ylim([1e-4, 1])

    ax = axes[0, 1]
    ax.plot(tp_data['time_hours'], tp_data['heart_rate'], 'r-', linewidth=2, label='Heart Rate')
    ax.set_xlabel('Time (hours)')
    ax.set_ylabel('Heart Rate (bpm)')
    ax.set_title(f'Vital Signs: Patient {tp_patient}')
    ax2 = ax.twinx()
    ax2.plot(tp_data['time_hours'], tp_data['bp_sys'], 'b-', linewidth=2, label='Systolic BP')
    ax2.set_ylabel('Blood Pressure (mmHg)', color='b')
    ax.legend(loc='upper left')
    ax2.legend(loc='upper right')
    ax.grid(True, alpha=0.3)

    # False Positive case
    fp_patient = fp_patients[0]
    fp_data = test_data[test_data['patient_id'] == fp_patient]

    ax = axes[1, 0]
    ax.plot(fp_data['time_hours'], fp_data['pred_proba'], 'b-', linewidth=2)
    ax.axhline(alert_thr, color='r', linestyle='--', label=f'Alert threshold ({alert_thr:.3f})')
    ax.set_xlabel('Time (hours)')
    ax.set_ylabel('Sepsis Probability')
    ax.set_title(f'False Positive: Patient {fp_patient} (No actual sepsis)')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_yscale('log'); ax.set_ylim([1e-4, 1])

    ax = axes[1, 1]
    ax.plot(fp_data['time_hours'], fp_data['heart_rate'], 'r-', linewidth=2, label='Heart Rate')
    ax.set_xlabel('Time (hours)')
    ax.set_ylabel('Heart Rate (bpm)')
    ax.set_title(f'Vital Signs: Patient {fp_patient}')
    ax2 = ax.twinx()
    ax2.plot(fp_data['time_hours'], fp_data['bp_sys'], 'b-', linewidth=2, label='Systolic BP')
    ax2.set_ylabel('Blood Pressure (mmHg)', color='b')
    ax.legend(loc='upper left')
    ax2.legend(loc='upper right')
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    print("\n🔍 OBSERVATION:")
    print("   Top row: True positive - model correctly identified deteriorating patient")
    print("   Bottom row: False positive - model triggered on patient who didn't develop sepsis")
    print("   \n   Both may show similar vital sign patterns, making discrimination challenging!")
else:
    print("Not enough positive cases in test set for visualization")


🔍 OBSERVATION:


   Top row: True positive - model correctly identified deteriorating patient
   Bottom row: False positive - model triggered on patient who didn't develop sepsis
   
   Both may show similar vital sign patterns, making discrimination challenging!


## 10. Key Takeaways

### What We Learned

1. **Feature engineering is critical**: Raw vital signs alone are insufficient. Temporal features (4 h slopes, deltas, 2/4/6 h rolling statistics) capture the gradual deterioration pattern of sepsis.

2. **High AUROC ≠ Clinical utility**:
   - Section 7b compares our model with the Chapter 3 system (82% sensitivity at 75% specificity) per admission, using an explicit alert policy: the synthetic task was calibrated to be about as hard
   - The unit of analysis matters: per 15-minute row far fewer cases are positive than per admission, so row-level sensitivity/specificity must not be combined with the 2% prevalence of admissions
   - But with ~2% sepsis prevalence, even 95% specificity yields many false positives (Chapter 3's 82%/75% model: ~15 false alarms per true alert)
   - This creates alert fatigue in clinical practice

3. **Threshold selection is a clinical decision**:
   - Lower threshold → Higher sensitivity, more false alarms
   - Higher threshold → Fewer false alarms, missed cases
   - Must involve clinicians and consider workflow

4. **Class imbalance matters**:
   - Sepsis is rare (~2% of admissions, Chapter 3)
   - Must use techniques like class weighting
   - Evaluate using appropriate metrics (PR curve, not just ROC)

5. **Temporal validation is essential** (this notebook only splits by patient):
   - Cannot shuffle time series data randomly
   - Must validate on future time periods
   - Performance typically degrades over time

### Connections to Book Chapters

- **Chapter 3 (Seven Journeys)**: Marcus's story provides the clinical motivation
- **Chapter 4 (Data Preparation)**: Feature engineering, handling missing data, temporal splitting
- **Chapter 5 (Evaluation)**: AUROC interpretation, PPV in imbalanced settings, threshold selection
- **Chapter 7 (Time Series)**: This chapter - temporal feature extraction, rolling statistics, gradient boosting
- **Chapter 10 (Interpretability)**: Feature importance analysis
- **Chapter 11 (Deployment)**: Alert fatigue, workflow integration, false positive rates

### Real-World Considerations

**The Epic Sepsis Model**: A widely deployed sepsis prediction system faced criticism for:
- 18:1 false positive ratio at some hospitals
- Declining clinician response rates due to alert fatigue
- Questions about mortality benefit

**For Marcus** (Journey 1): The model was right: it flagged him at 2:47 AM with a 73% risk. But it was the 13th alert of the night shift, most earlier ones had been false alarms, and he deteriorated about 45 minutes later before anyone checked. He spent 3 days in the ICU. A correct prediction only helps if the alert burden lets clinicians act on it (compare the false alarms per detected sepsis printed in Section 7b for our synthetic model).

### Next Steps in Practice

1. **Multi-institutional validation**: Does the model work at other hospitals?
2. **Prospective clinical trial**: Does it actually improve outcomes?
3. **Workflow integration**: How to present alerts to minimize cognitive burden?
4. **Continuous monitoring**: Performance may degrade over time
5. **Ethical considerations**: Who is responsible when the model is wrong?

---

## Exercises

1. **Experiment with features**: Try removing temporal features and use only current vital signs. How much does performance degrade?

2. **Cost-sensitive learning**: Assign different costs to false positives vs false negatives. How does this change the optimal threshold?

3. **Feature selection**: The model has 100+ features. Can you identify the top 20 and retrain with similar performance?

4. **Temporal validation**: Implement proper time-series cross-validation. How does performance vary across folds?

5. **Calibration**: Plot a calibration curve. Are the predicted probabilities well-calibrated? (See Chapter 5)

---

*This notebook is part of "AI in Healthcare" (Volume 1, Chapter 7: Time Series Analysis)*  
*This implements Journey 1 (Marcus - Sepsis) from Chapter 3*  
*For clinical context, see Chapter 3. For time series methods, see Chapter 7.*